# Sociolinguistics : observing linguistic variation in social context

In this Assignment we want to focus on a task you might be interested in if you are studying sociolinguistics or social variation in language use. We focus on language variation in relation to specific identity marker: age. We already manually annotated some data and found some potential lexical choices or grammatical constructions that might be more frequent in younger versus older speakers. 

To have some data to study the differences in language between people of younger versus older age groups we need texts with speaker information. As an approximation we use two different Reddit communities (sub-reddits) that are focused on different age groups:
- r/GenZ (a community for members of Generation Z)
- r/AskOldPeople (a community for older adults to share their experiences and advice)

Of course we cannot be sure that all members of these communities belong to the respective age groups, but we can assume that there will be a lot of overlap.

The goal of this assignment is to find out differences in language use between these two communities.

As practical methods we want to learn:
- how to add additional information to an existing corpus object
- how to extract linguistic features that are useful to study language variation
- how to compare the two communities based on these features

## Analyze differences between age groups

Now that we have added all additional information to the utterance objects in the two corpora, we can start analyzing the differences between the two age groups. We can retrieve the utterance dataframes and compare the feature values between the two corpora. 

*Task 1*: 

For each feature, compare the distribution of values between the two corpora. You can use boxplots or violin plots to visualize the distributions. They can show you the mean and variance of each feature in each corpus. You can use the matplotlib library and or the seaborn library that come with pre-defined functions to create such a plot. 

Of course you can also use other methods to analyze the different features (e.g use statistical tests to compare whether the feature values significantly differ between the two corpora.)

- (1) conduct the analysis for yourself. if you feel you are ready to share some insights go to a free spot at the group table. 
- (2) once you have a big enough group (>=2 people) discuss and compare your findings. When you are done with discussing them, add anything that you feel is missing to the result padlet (https://padlet.com/kbm54vknmk/results-assignment-sociolinguistics-gpai0io3rwyaub7z)
 
**Which features show the biggest differences between the two age groups? What do these differences tell you about the language use of younger versus older speakers?**


## Part 2 of the Assignment:


## Starting Point
You should now have two processed and filtered corpora:
- genz-filtered : containing utterances from the r/GenZ subreddit with a certain number of tokens, linguistic features and the number of gen-z specific word occurrences added as meta information
- ask-old-people-filtered : containing utterances from the r/AskOldPeople subreddit with a certain number of tokens, linguistic features and the number of gen-z specific word occurrences added as meta information

Our goal is now to analyze the differences between these two corpora based on the linguistic features we extracted.
I have pre-processed and enriched the two corpora as discussed. They are stored in "additional_data/genz-filtered" and "additional_data/ask-old-people-filtered". Load them both and store them into a variable with corpus_old_current = Corpus(filename="path/to/corpus").


In [ ]:
from convokit import Corpus

corpus_genz = Corpus(filename="/Users/falkne/PycharmProjects/nlpforSocialInteractions/additional_data/gen-z-filtered")
corpus_old = Corpus(
    filename="/Users/falkne/PycharmProjects/nlpforSocialInteractions/additional_data/ask-old-people-filtered")


In [ ]:
corpus_old.print_summary_stats()

In [ ]:
dataframe_genz = corpus_genz.get_utterances_dataframe()
dataframe_old = corpus_old.get_utterances_dataframe()

In [ ]:
dataframe_genz.head()

plot function to create boxplot, violin plot or histogram. one can specify the plot type and input arguments need to be the utterance dataframe (one for each corpus) and the column name of the feature to plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


def plot_feature(utterance_df_young, utterance_df_old, feature_col, plot_type="box"):
    # Prepare tidy long-format data
    data = pd.concat([
        pd.DataFrame({
            "Group": "Gen Z",
            "Value": utterance_df_young[feature_col]
        }),
        pd.DataFrame({
            "Group": "AskOldPeople",
            "Value": utterance_df_old[feature_col]
        })
    ], ignore_index=True)

    if plot_type == "box":
        plt.figure(figsize=(8, 6))
        sns.boxplot(x='Group', y='Value', data=data, palette=["green", "orange"])
        plt.title(f"Boxplot of {feature_col} by Group")
    elif plot_type == "violin":
       # plt.figure(figsize=(8, 6))
        sns.violinplot(x='Group', y='Value', data=data, palette=["green", "orange"], linewidth=1.2, inner="box")
        plt.title(f"Violin Plot of {feature_col} by Group")
    else:
        sns.histplot(x='Value', hue='Group', data=data, bins=50, kde=True, stat="density", palette=["green", "orange"],
                     alpha=0.6, common_norm=False)
        plt.title(f"Histogram of {feature_col} by Group")

    plt.ylabel(feature_col)
    plt.xlabel("Group")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_feature(dataframe_genz, dataframe_old, "meta.n_pron", plot_type="box")

Statistical test to compare whether distributions of a specific feature significantly differ. The Mann-Whitney U test does not make assumptions about the distribution of the data and is suitable for comparing two independent samples. The p-value will indicate whether the difference between the two groups is statistically significant. Additionally, we calculate the rank-biserial correlation as an effect size measure to quantify the magnitude of the difference between the two groups.

In [ ]:
from scipy.stats import mannwhitneyu

x = dataframe_genz['meta.n_unoun'].tolist()
y = dataframe_old['meta.n_unoun'].tolist()
U, p = mannwhitneyu(x, y, alternative='two-sided')
# and effect size
rank_biserial_corr = 1 - (2 * U / (len(x) * len(y)))
# print .3f
print(f"Mann-Whitney U test: U={U}, p={p:.3f}")
print(f"Rank-biserial correlation: {rank_biserial_corr:.3f}")

We can use this tool from Convokit to identify words that are significantly more frequent in one corpus compared to the other. This can help us identify lexical differences between the two age groups. You can also change the ngram_range parameter to look at bigrams or trigrams instead of unigrams. You can use stop_words='english' to ignore common English words that might not be informative. 

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from convokit import FightingWords

cv = CountVectorizer(ngram_range=(1, 1), stop_words='english')
# import Counter
fw = FightingWords(ngram_range=(1, 1), cv=cv)

In [ ]:
# add subreddit info to every meta
for utt in corpus_genz.iter_utterances():
    utt.meta['subreddit'] = 'genZ'
for utt in corpus_old.iter_utterances():
    utt.meta['subreddit'] = 'askoldpeople'

In [ ]:
all_utterances = []
for utt in corpus_genz.iter_utterances():
    all_utterances.append(utt)
for utt in corpus_old.iter_utterances():
    all_utterances.append(utt)
# create one combined corpus object
combined_corpus = Corpus(utterances=all_utterances)

In [ ]:
fw.fit(combined_corpus, class1_func=lambda utt: utt.meta['subreddit'] == 'genZ',
       class2_func=lambda utt: utt.meta['subreddit'] == "askoldpeople")

In [ ]:
df = fw.summarize(combined_corpus, plot=True, class1_name='r/genz', class2_name='r/askoldpeople')

In [ ]:
fw.transform(combined_corpus, config={'annot_method': 'top_k', 'top_k': 10})

In [ ]:
# this shows the top 10 distinct gen z words
list(fw.get_top_k_ngrams()[0])

In [ ]:
# and this the ones for askoldpeople
list(fw.get_top_k_ngrams()[1])

In [ ]:
# here we can see some examples of utterances with askoldpeople words
counter = 0
while counter <10:
    for utt in combined_corpus.iter_utterances():
        if len(utt.meta['fighting_words_class2']) > 7 and len(utt.meta['fighting_words_class1']) < 1:
            print(utt.meta['subreddit'])
            print(utt.meta['fighting_words_class1'])
            print(utt.meta['fighting_words_class2'])
            print(utt.text)
            counter += 1



## Analyze differences at the speaker level

Instead of analyzing the differences at the utterance level, we can also aggregate the feature values at the speaker level. This way, we can see how individual speakers differ in their language use.

*Task 2*: For each speaker in each corpus, calculate the average value for each feature across all their utterances. Create a new dataframe or dictionary for each corpus that contains the speaker ids as keys and the average feature values as values. Remove speakers that have less than 10 utterances to ensure reliable estimates. As a result each speaker can be represented as a feature vector.
 Use k-means clustering to cluster the speakers within each corpus. How many clusters seem to be appropriate for each corpus? Do the clusters correspond to different styles of language use? Can you interpret the clusters in terms of linguistic features?
 (Hint: you can for example investigate the cluster centroids to see which features are most prominent in each cluster, e.g. plot the feature vectors of the centroids as bar plots or a heatmap.)

In [ ]:
dataframe_old.head()

One possible approach to this question in clustering. We can aggregate the feature values at the speaker level and then use k-means clustering to group speakers based on their linguistic features. This way we can identify different styles of language use within each age group. We can use k-means clustering from the sklearn library.

In [ ]:
# get all the feature columns
meta_cols = [col for col in dataframe_old.columns if col.startswith('meta.n')]
extra_feats = ['meta.a_word_ps', 'meta.a_bry_ps', 'meta.corr_ttr', 'meta.num_genz_words']
meta_cols = meta_cols + extra_feats
meta_cols


In [ ]:
# first create a dictionary to store the feature vectors for each speaker. only include speakers with at least 10 utterances.
speakerid2feature_vector_young = {}
# for each speaker we want to create a vector of the meta columns. 
for speaker in corpus_genz.iter_speakers():
    # get all utterances by this speaker
    utterances_by_this_speaker = dataframe_genz[dataframe_genz['speaker'] == speaker.id]
    # check if len(utterances_by_this_speaker) >=20
    if len(utterances_by_this_speaker) >=10:
       # get only meta cols
       features = utterances_by_this_speaker[meta_cols]
       # get the mean
       feature_vec = features.mean()
       # convert to a vector
       vector = feature_vec.values
       # add this vector to "vectors" of the Speaker object
       speakerid2feature_vector_young[speaker.id] = vector

In [ ]:
speakerid2feature_vector_old = {}
for speaker in corpus_old.iter_speakers():
    utterances_by_this_speaker = dataframe_old[dataframe_old['speaker'] == speaker.id]
    if len(utterances_by_this_speaker) >=10:
       features = utterances_by_this_speaker[meta_cols]
       feature_vec = features.mean()
       vector = feature_vec.values
       speakerid2feature_vector_old[speaker.id] = vector

In [ ]:
# this method applied k-means clustering to cluster the speakers based on their feature vectors. it returns a dictionary mapping speaker ids to their assigned cluster labels.
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
def cluster_speakers(speakerid2feature_vector, n_clusters=5):
    speaker_ids = list(speakerid2feature_vector.keys())
    feature_matrix = list(speakerid2feature_vector.values())
    # standardize the feature matrix 
    feature_matrix = scaler.fit_transform(feature_matrix)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(feature_matrix)
    centroids = kmeans.cluster_centers_
    
    speakerid2cluster = {speaker_id: kmeans.labels_[i] for i, speaker_id in enumerate(speaker_ids)}
    return speakerid2cluster, centroids

In [ ]:
young2cluster, centroids_young = cluster_speakers(speakerid2feature_vector_young, n_clusters=4)

In [ ]:
old2cluster, centroids_old = cluster_speakers(speakerid2feature_vector_old, n_clusters=5)

In [ ]:
# plot a countplot of the cluster assignments to see the distribution of speakers across clusters
import matplotlib.pyplot as plt
sns.countplot(x=list(young2cluster.values()))
plt.title("Distribution of Speakers across Clusters (Gen Z)")
plt.xlabel("Cluster ID")
plt.ylabel("Number of Speakers")
plt.show()

In [ ]:
sns.countplot(x=list(old2cluster.values()))
plt.title("Distribution of Speakers across Clusters (AskOldPeople)")
plt.xlabel("Cluster ID")
plt.ylabel("Number of Speakers")
plt.show()

Now we want to get and idea of the characteristics of each cluster. We can retrieve the cluster centroids and analyze the feature values for each cluster.

In [ ]:
# plot the centroids as a heatmap for the young speakers
import numpy as np
plt.figure(figsize=(6, 16))
sns.heatmap(centroids_young.T, annot=True, fmt=".1f", cmap="YlGnBu",
            yticklabels=meta_cols, 
            xticklabels=[f"Cluster {i}" for i in range(centroids_young.shape[0])])
plt.title("Cluster Centroids for Gen Z Speakers")
plt.xlabel("Features")
plt.ylabel("Clusters")
plt.show()


In [ ]:
# plot the centroids as a heatmap for the young speakers
plt.figure(figsize=(6, 16))
sns.heatmap(centroids_old.T, annot=True, fmt=".1f", cmap="YlGnBu",
            yticklabels=meta_cols, 
            xticklabels=[f"Cluster {i}" for i in range(centroids_young.shape[0])])
plt.title("Cluster Centroids for AskOldPeople Speakers")
plt.xlabel("Features")
plt.ylabel("Clusters")
plt.show()